In [1]:
import pickle
import functools
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import networkx as nx
import pyproj
import geopandas as gpd
import shapely
import pandas as pd
from shapely import Polygon, Point, LineString

import datetime
import simpy
import opentnsim
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core.plotutils import generate_vessel_gantt_chart
from opentnsim.graph import mixins as graph_module

from shapely.ops import transform
from pyproj import Transformer
from sklearn.cluster import KMeans

import pyarrow as pa
import movingpandas as mpd
from lonboard import Map, PathLayer, viz
from lonboard.colormap import apply_categorical_cmap
from lonboard import TripsLayer

from tqdm.auto import tqdm

from pathlib import Path

In [2]:
src_dir = Path('~').expanduser() / 'data/d-osp/gtsm'

In [3]:
def get_neighbors(ds, face_id):
    Mesh_face_nodes = ds['Mesh_face_nodes'].values - 1
    face_nodes = Mesh_face_nodes[face_id, :]
    face_nodes = face_nodes[~np.isnan(face_nodes)]
    face_nodes = face_nodes[face_nodes != -2]

    # Build node-to-face mapping
    node_to_faces = {}
    for f_id, nodes in enumerate(Mesh_face_nodes):
        valid_nodes = nodes[~np.isnan(nodes)]
        for node in valid_nodes:
            node_to_faces.setdefault(node, set()).add(f_id)

    # Collect neighbors
    neighbors = set()
    for node in face_nodes:
        neighbors.update(node_to_faces[node])

    neighbors.discard(face_id)
    return sorted(neighbors)

def build_edge_gdf(node_gdf):
    edge_gdf = {'source': [], 'target': [], 'geometry': []}
    for idx, node in tqdm(node_gdf.iterrows(), total=len(node_gdf)):
        source_node = idx
        target_nodes = get_neighbors(ds, source_node)
        for target_node in target_nodes:
            if target_node in edge_gdf['source']:
                continue
            geometry = LineString([node_gdf.iloc[source_node]['geometry'], node_gdf.iloc[target_node]['geometry']])
            
            edge_gdf['source'].append(source_node)
            edge_gdf['target'].append(target_node)
            edge_gdf['geometry'].append(geometry)

    edge_gdf = gpd.GeoDataFrame(edge_gdf)
    edge_gdf['edge_id'] = np.arange(len(edge_gdf))
    edge_gdf = edge_gdf.set_crs('EPSG:4326')
    edge_gdf = edge_gdf.to_crs('EPSG:3035')
    edge_gdf['length_m'] = shapely.length(edge_gdf['geometry'])
    return edge_gdf

def compute_edge_values(node_gdf, edges_static):
    # Extract node-based values
    cu = node_gdf["currents_u"].to_numpy()
    cv = node_gdf["currents_v"].to_numpy()

    # Build an edge dataframe with new values only
    df = pd.DataFrame({
        "edge_id": edges_static["edge_id"],
        "currents_u": (cu[edges_static["source"].values] + cu[edges_static["target"].values]) / 2,
        "currents_v": (cv[edges_static["source"].values] + cv[edges_static["target"].values]) / 2,
    })

    # Merge back into a GeoDataFrame
    return edges_static.merge(df, on="edge_id")


def create_nodes_and_edges_from_ds(ds, edge_gdf_static, timestep):
    ds_t = ds.sel({"time": timestep}, method="nearest")
    node_geoms = gpd.points_from_xy(ds_t['Mesh_face_x'], ds_t['Mesh_face_y'])
    node_gdf = gpd.GeoDataFrame({'currents_u': ds_t['currents_u'], 'currents_v': ds_t['currents_v'], 'geometry': node_geoms})

    edge_gdf = compute_edge_values(node_gdf=node_gdf, edges_static=edge_gdf_static)
    node_gdf = node_gdf.set_crs("EPSG:4326")
    node_gdf = node_gdf.to_crs("EPSG:3035")
    return node_gdf, edge_gdf


def create_graph(node_gdf, edge_gdf):
    G = nx.from_pandas_edgelist(edge_gdf, edge_attr=True)
    node_attributes = node_gdf.to_dict('index')
    nx.set_node_attributes(G, node_attributes)

    src_crs = "EPSG:3035"
    dst_crs = "EPSG:4326"

    # Build a transformer. always_xy=True ensures (lon, lat) order.
    project = Transformer.from_crs(src_crs, dst_crs, always_xy=True).transform
    G = add_node_xy(G)
    # G = add_edge_geometries(G)
    # G = add_node_geometries(G, project=project)
    return G

def create_coarse_graph(node_gdf, edge_gdf):
    G = nx.from_pandas_edgelist(edge_gdf, edge_attr=True)
    node_attributes = node_gdf.to_dict('index')
    nx.set_node_attributes(G, node_attributes)

    G2 = coarsen_mesh_spatial(G, 500)

    src_crs = "EPSG:3035"
    dst_crs = "EPSG:4326"

    # Build a transformer. always_xy=True ensures (lon, lat) order.
    project = Transformer.from_crs(src_crs, dst_crs, always_xy=True).transform
    G2 = add_edge_geometries(G2)
    G2 = add_node_geometries(G2, project=project)
    return G2

def add_node_xy(G):
    for u in G.nodes():
        x, y = G.nodes[u]['geometry'].x, G.nodes[u]['geometry'].y
        G.nodes[u]['x'] = x
        G.nodes[u]['y'] = y
    return G

def add_edge_geometries(G):
    for u, v in G.edges():
        x1, y1 = G.nodes[u]['x'], G.nodes[u]['y']
        x2, y2 = G.nodes[v]['x'], G.nodes[v]['y']
        G.edges[u, v]['geometry'] = LineString([(x1, y1), (x2, y2)])
        G.edges[u, v]['length_m'] = shapely.length(G.edges[u, v]['geometry'])
    return G

def add_node_geometries(G, project):
    for u in G.nodes():
        x, y = G.nodes[u]['x'], G.nodes[u]['y']
        geometry = shapely.Point([x, y])
        geometry = transform(project, geometry)

        G.nodes[u]['geometry'] = geometry

    return G


def graph_edges_to_gdf(G):
    edge_data = []
    for u, v, data in G.edges(data=True):
        # Get coordinates of the two super-nodes
        x1, y1 = G.nodes[u]['x'], G.nodes[u]['y']
        x2, y2 = G.nodes[v]['x'], G.nodes[v]['y']
        # Create LineString geometry
        geom = LineString([(x1, y1), (x2, y2)])
        # Collect attributes
        edge_data.append({
            'source': u,
            'target': v,
            'currents_u': data.get('currents_u'),
            'currents_v': data.get('currents_v'),
            'length_m': data.get('length'),
            # 'edge_id': data.get('edge_id'),
            'geometry': geom
        })
    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(edge_data, geometry='geometry', crs="EPSG:3035")  # or your CRS
    return gdf

def compute_direction(p1, p2):
    x1, y1 = p1
    x2, y2 = p2

    # Direction unit vector
    dx, dy = x2 - x1, y2 - y1
    length = np.sqrt(dx**2 + dy**2)
    if length == 0:
        direction = 0
    else:
        direction = (dx/length, dy/length)

    return direction


def coarsen_mesh_spatial(G, target_clusters):
    # Extract coordinates
    coords = np.array([[G.nodes[n]['geometry'].x, G.nodes[n]['geometry'].y] for n in G.nodes()])
    node_ids = list(G.nodes())

    # Cluster nodes using KMeans
    kmeans = KMeans(n_clusters=target_clusters, random_state=42)
    labels = kmeans.fit_predict(coords)

    # Create new graph
    H = nx.Graph()

    # Add super-nodes with averaged coordinates
    for cluster_id in range(target_clusters):
        cluster_nodes = [node_ids[i] for i in range(len(node_ids)) if labels[i] == cluster_id]
        avg_x = np.mean([G.nodes[n]['geometry'].x for n in cluster_nodes])
        avg_y = np.mean([G.nodes[n]['geometry'].y for n in cluster_nodes])
        H.add_node(cluster_id, x=avg_x, y=avg_y, members=cluster_nodes)

    # Build edges between clusters and average attributes
    edge_dict = {}
    for u, v, data in G.edges(data=True):
        cu = labels[node_ids.index(u)]
        cv = labels[node_ids.index(v)]
        if cu != cv:
            key = tuple(sorted((cu, cv)))
            if key not in edge_dict:
                edge_dict[key] = {'currents_u': [], 'currents_v': []}
            edge_dict[key]['currents_u'].append(data['currents_u'])
            edge_dict[key]['currents_v'].append(data['currents_v'])

    # Add averaged edges to new graph
    for (cu, cv), attrs in edge_dict.items():
        avg_currents_u = np.mean(attrs['currents_u'])
        avg_currents_v = np.mean(attrs['currents_v'])
        H.add_edge(cu, cv, currents_u=avg_currents_u, currents_v=avg_currents_v)

    return H

def create_digraph(G):
    src_crs = "EPSG:3035"
    dst_crs = "EPSG:4326"

    # Build a transformer. always_xy=True ensures (lon, lat) order.
    project = Transformer.from_crs(src_crs, dst_crs, always_xy=True).transform
    G2 = nx.DiGraph()
    for u, v, data in G.edges(data=True):
        # Node positions
        x1, y1 = G.nodes[u]['geometry'].x, G.nodes[u]['geometry'].y
        x2, y2 = G.nodes[v]['geometry'].x, G.nodes[v]['geometry'].y

        direction = compute_direction((x1, y1), (x2, y2))
        direction_u, direction_v = direction

        data['direction_u'] = direction_u
        data['direction_v'] = direction_v

        if u not in G2.nodes:
            G2.add_node(u, **G.nodes[u])
        if v not in G2.nodes:
            G2.add_node(v, **G.nodes[v])

        current_u = data['currents_u']
        current_v = data['currents_v']

        current = np.dot([current_u, current_v], [direction_u, direction_v])
        data['Info'] = {'Current': current}

        geometry = data['geometry']
        geometry = transform(project, geometry)
        data['geometry'] = geometry

        G2.add_edge(u, v, **data)

    for v, u, data in G.edges(data=True):
        # Node positions
        x1, y1 = G.nodes[u]['geometry'].x, G.nodes[u]['geometry'].y
        x2, y2 = G.nodes[v]['geometry'].x, G.nodes[v]['geometry'].y
            
        direction = compute_direction((x1, y1), (x2, y2))
        direction_u, direction_v = direction

        data['direction_u'] = direction_u
        data['direction_v'] = direction_v

        current_u = data['currents_u']
        current_v = data['currents_v']

        current = np.dot([current_u, current_v], [direction_u, direction_v])
        data['Info'] = {'Current': current}

        geometry = data['geometry']
        geometry = transform(project, geometry)
        data['geometry'] = geometry

        G2.add_edge(u, v, **data)

    return G2


def create_gtsm_digraph(ds, edge_gdf_static, timestep):
    node_gdf, edge_gdf = create_nodes_and_edges_from_ds(ds=ds, edge_gdf_static=edge_gdf_static, timestep=timestep)
    G = create_graph(node_gdf=node_gdf, edge_gdf=edge_gdf)
    G_d = create_digraph(G)
    return G_d

In [4]:
edge_gdf_static = gpd.read_file(src_dir / "static-gtsm-edges.gpkg")
ds = xr.open_dataset(src_dir / 'filtered-currents.nc')

time_start = ds.time.values[0]

G = create_gtsm_digraph(ds=ds, edge_gdf_static=edge_gdf_static, timestep=time_start)

In [5]:
# make your preferred Vessel class out of available mix-ins.
Vessel = type(
    "Vessel", 
    (
        opentnsim.core.Identifiable, # allows to give the object a name and a random ID,
        opentnsim.core.Movable,      # allows the object to move, with a fixed speed, while logging this activity
    ), 
    {}
)

In [6]:
def __compute_weight(
    origin, target, dictionary_edge, ship_velocity
):
    # order classes from smallest to largest
    # if dictionary_edge["is_border"]:
    #     return dictionary_edge["length_m"]
            
    edge_length = dictionary_edge["length_m"]
    edge_current = dictionary_edge['Info']["Current"]
    ship_velocity += edge_current

    taken_time = edge_length / ship_velocity
    return taken_time


def path_corrected(
    graph, origin, destination, ship_velocity
):
    """find a path restricted to allowed cemt classes

    Parameters
    ----------
    graph : networkx.Graph
        graph in which to find a path. graph edges should have information 'cemt'.
    origin : str
        origin node id
    destination : str
        destination node id
    ship_cemt_classe : str
        cemt class of the ship.
    """
    # define order of cemt classes

    # create function to compute weights for this ship
    compute_weight = functools.partial(
        __compute_weight,
        ship_velocity=ship_velocity
    )

    # find the path
    path = nx.dijkstra_path(graph, origin, destination, weight=compute_weight)
    return path

In [7]:
def change_current(node, vessel):
    env = vessel.env

    current_time = pd.Timestamp(env.now, unit='s')
    current_graph = create_gtsm_digraph(ds=ds, edge_gdf_static=edge_gdf_static, timestep=current_time)
    env.graph = current_graph
    
    vessel_position = vessel.current_node
    target_node = vessel.route[-1]

    try:
        idx = vessel.route.index(vessel_position)
    except ValueError:
        # Safety fallback: if not found, treat as full reroute
        idx = 0

    sailed_route = vessel.route[:idx + 1]

    new_route = path_corrected(current_graph, vessel_position, target_node, ship_velocity=3)
    vessel.route = sailed_route[:-1] + new_route

    yield env.timeout(0)

In [8]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel. 
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

In [17]:
simulation_start = ds.time.values[0]
env = simpy.Environment(initial_time=simulation_start.astype(float) / 1e9)
env.epoch = simulation_start

env.graph = G

origin = 11693
destiniation = 7935
# create vessel from a dict 
path = path_corrected(graph=G, origin=origin, destination=destiniation, ship_velocity=3)

data_vessel = {
    "env": env,                                       # needed for simpy simulation
    "name": "Vessel",                                 # required by Identifiable
    "geometry": env.graph.nodes[path[0]]['geometry'], # required by Locatable
    "route": path,                                    # required by Routeable
    "v": 3,                                           # required by Movable, 1 m/s to check if the distance is covered in the expected time
}  # 

# create an instance of the Vessel class using the input dict data_vessel
vessel = Vessel(**data_vessel)
on_pass_node = functools.partial(change_current, vessel=vessel)
vessel.on_pass_node_functions = [on_pass_node]

# start the simulation
env.process(mission(env, vessel))
env.run()

In [19]:
# load the logbook data into a dataframe
df_adjusted = pd.DataFrame.from_dict(vessel.logbook)

print("'{}' logbook data:".format(vessel.name))  
print('')

display(df_adjusted)

trip_distance = graph_module.calculate_distance_along_path(G, vessel.route)
trip_duration = datetime.timedelta.total_seconds(vessel.logbook[-1]['Timestamp'] - vessel.logbook[0]['Timestamp'])

print("'{}' travelled a distance of {:.1f} meters".format(vessel.name, trip_distance))
print("'{}' took {:.1f} seconds to arrive at its destination".format(vessel.name, trip_duration))  
print("'{}' travelled at an average speed of {:.1f} meters per second".format(vessel.name, trip_distance/trip_duration))
print('')
print('')

'Vessel' logbook data:



,Message,Timestamp,Value,Geometry
0,Sailing from node 11693 to node 11773 start,2025-11-08 13:00:00.000000,0.000000,POINT (3912404.019177241 3208112.894432935)
1,Sailing from node 11693 to node 11773 stop,2025-11-08 13:09:09.366663,1630.688468,POINT (3912536.6272703195 3209738.182091775)
2,Sailing from node 11773 to node 11854 start,2025-11-08 13:09:09.366663,1630.688468,POINT (3912536.6272703195 3209738.182091775)
3,Sailing from node 11773 to node 11854 stop,2025-11-08 13:18:18.249460,3261.381116,POINT (3912669.2556425175 3211363.472288893)
4,Sailing from node 11854 to node 12022 start,2025-11-08 13:18:18.249460,3261.381116,POINT (3912669.2556425175 3211363.472288893)
...,...,...,...,...
155,Sailing from node 8218 to node 8106 stop,2025-11-09 01:56:44.260982,141495.327839,POINT (3968899.8604117027 3331334.2548301206)
156,Sailing from node 8106 to node 8048 start,2025-11-09 01:56:44.260982,141495.327839,POINT (3968899.8604117027 3331334.2548301206)
157,Sailing from node 8106 to node 8048 stop,2025-11-09 02:13:30.202432,143400.145493,POINT (3969999.257381352 3332889.779371133)
158,Sailing from node 8048 to node 7935 start,2025-11-09 02:13:30.202432,143400.145493,POINT (3969999.257381352 3332889.779371133)


'Vessel' travelled a distance of nan meters
'Vessel' took 48557.0 seconds to arrive at its destination
'Vessel' travelled at an average speed of nan meters per second




In [20]:
simulation_start = ds.time.values[0]
env = simpy.Environment(initial_time=simulation_start.astype(float) / 1e9)
env.epoch = simulation_start

env.graph = G

origin = 11693
destiniation = 7935
# create vessel from a dict 
path = path_corrected(graph=G, origin=origin, destination=destiniation, ship_velocity=3)

data_vessel = {
    "env": env,                                       # needed for simpy simulation
    "name": "Vessel",                                 # required by Identifiable
    "geometry": env.graph.nodes[path[0]]['geometry'], # required by Locatable
    "route": path,                                    # required by Routeable
    "v": 3,                                           # required by Movable, 1 m/s to check if the distance is covered in the expected time
}  # 

# create an instance of the Vessel class using the input dict data_vessel
vessel = Vessel(**data_vessel)

# start the simulation
env.process(mission(env, vessel))
env.run()

# load the logbook data into a dataframe
df_normal = pd.DataFrame.from_dict(vessel.logbook)

print("'{}' logbook data:".format(vessel.name))  
print('')

display(df_normal)

trip_distance = graph_module.calculate_distance_along_path(G, vessel.route)
trip_duration = datetime.timedelta.total_seconds(vessel.logbook[-1]['Timestamp'] - vessel.logbook[0]['Timestamp'])

print("'{}' travelled a distance of {:.1f} meters".format(vessel.name, trip_distance))
print("'{}' took {:.1f} seconds to arrive at its destination".format(vessel.name, trip_duration))  
print("'{}' travelled at an average speed of {:.1f} meters per second".format(vessel.name, trip_distance/trip_duration))
print('')
print('')

'Vessel' logbook data:



,Message,Timestamp,Value,Geometry
0,Sailing from node 11693 to node 11773 start,2025-11-08 13:00:00.000000,0.000000,POINT (3912404.019177241 3208112.894432935)
1,Sailing from node 11693 to node 11773 stop,2025-11-08 13:09:09.366663,1630.688468,POINT (3912536.6272703195 3209738.182091775)
2,Sailing from node 11773 to node 11854 start,2025-11-08 13:09:09.366663,1630.688468,POINT (3912536.6272703195 3209738.182091775)
3,Sailing from node 11773 to node 11854 stop,2025-11-08 13:18:18.249460,3261.381116,POINT (3912669.2556425175 3211363.472288893)
4,Sailing from node 11854 to node 12022 start,2025-11-08 13:18:18.249460,3261.381116,POINT (3912669.2556425175 3211363.472288893)
...,...,...,...,...
171,Sailing from node 8218 to node 8106 stop,2025-11-09 03:28:18.481384,147337.207813,POINT (3968899.8604117027 3331334.2548301206)
172,Sailing from node 8106 to node 8048 start,2025-11-09 03:28:18.481384,147337.207813,POINT (3968899.8604117027 3331334.2548301206)
173,Sailing from node 8106 to node 8048 stop,2025-11-09 03:45:23.683893,149242.025467,POINT (3969999.257381352 3332889.779371133)
174,Sailing from node 8048 to node 7935 start,2025-11-09 03:45:23.683893,149242.025467,POINT (3969999.257381352 3332889.779371133)


'Vessel' travelled a distance of nan meters
'Vessel' took 54084.2 seconds to arrive at its destination
'Vessel' travelled at an average speed of nan meters per second




In [24]:
# Get the vessel log and convert it to a GeoDataFrame
vessel_log_gdf_adjusted = gpd.GeoDataFrame(df_adjusted, crs="EPSG:3035", geometry='Geometry')
vessel_log_gdf_adjusted = vessel_log_gdf_adjusted.to_crs('EPSG:4326')
vessel_log_gdf_adjusted['t'] = vessel_log_gdf_adjusted["Timestamp"]
# Create a Trajectory using movingpandas with the id being the ship name.
traj_adjusted = mpd.Trajectory(vessel_log_gdf_adjusted, t="t", traj_id="ship_1")
# Convert Trajectory to GeoDataFrame to save to a .gpkg file
traj_gdf_adjusted = traj_adjusted.to_point_gdf()

# Get the vessel log and convert it to a GeoDataFrame
vessel_log_gdf_normal = gpd.GeoDataFrame(df_normal, crs="EPSG:3035", geometry='Geometry')
vessel_log_gdf_normal = vessel_log_gdf_normal.to_crs('EPSG:4326')
vessel_log_gdf_normal['t'] = vessel_log_gdf_normal["Timestamp"]
# Create a Trajectory using movingpandas with the id being the ship name.
traj_normal = mpd.Trajectory(vessel_log_gdf_normal, t="t", traj_id="ship_2")
# Convert Trajectory to GeoDataFrame to save to a .gpkg file
traj_gdf_normal = traj_normal.to_point_gdf()

traj_gdf_adjusted

,Message,Timestamp,Value,Geometry,traj_id
t,,,,,
2025-11-08 13:00:00.000000,Sailing from node 11693 to node 11773 start,2025-11-08 13:00:00.000000,0.000000,POINT (4.06494 51.8335),ship_1
2025-11-08 13:09:09.366663,Sailing from node 11693 to node 11773 stop,2025-11-08 13:09:09.366663,1630.688468,POINT (4.06494 51.84814),ship_1
2025-11-08 13:18:18.249460,Sailing from node 11773 to node 11854 stop,2025-11-08 13:18:18.249460,3261.381116,POINT (4.06494 51.86279),ship_1
2025-11-08 13:29:07.658438,Sailing from node 11854 to node 12022 stop,2025-11-08 13:29:07.658438,5178.706174,POINT (4.05029 51.87744),ship_1
2025-11-08 13:40:03.647983,Sailing from node 12022 to node 12199 stop,2025-11-08 13:40:03.647983,7095.857946,POINT (4.03564 51.89209),ship_1
...,...,...,...,...,...
2025-11-09 01:25:55.235743,Sailing from node 8389 to node 8273 stop,2025-11-09 01:25:55.235743,137685.193081,POINT (4.72412 52.94678),ship_1
2025-11-09 01:40:56.852465,Sailing from node 8273 to node 8218 stop,2025-11-09 01:40:56.852465,139590.343687,POINT (4.73877 52.96143),ship_1
2025-11-09 01:56:44.260982,Sailing from node 8218 to node 8106 stop,2025-11-09 01:56:44.260982,141495.327839,POINT (4.75342 52.97607),ship_1


In [25]:
# Visualize the sailed route on the map
viz([traj_gdf_adjusted, traj_gdf_normal])

In [27]:
traj_gdf_combined = pd.concat([traj_gdf_adjusted, traj_gdf_normal])
# For the animation we need to create a TrajectoryCollection first. Here we construct it again from the point_gdf
traj_collection = mpd.TrajectoryCollection(traj_gdf_combined, traj_id_col="traj_id", t="t")

# Now we only have one ship, but you can assign different colors to different ships
trajid_to_color = {
    "ship_1": [0, 122, 255, 255],
    "ship_2": [255, 122, 0, 255],
}

get_color = apply_categorical_cmap(
    pa.array(traj_gdf_combined["traj_id"].unique()), trajid_to_color
)

# Create the visualization from the TrajectoryCollection
trips_layer = TripsLayer.from_movingpandas(
    traj_collection, width_min_pixels=5, trail_length=10 * 60 * 1000, get_color=get_color
)

/Users/hemert/projects/d-osp/criticality-analysis/.venv/lib/python3.12/site-packages/lonboard/traits/_timestamp.py:152: UserWarning:

Reducing precision of input timestamp data to 's' to fit into available GPU precision.



In [28]:
# Show the visualization on the map
m = Map(trips_layer, height=600)
m

In [30]:

# Animate the visualization
trips_layer.animate(step=datetime.timedelta(minutes=1), fps=30)